# Memory Information

In [ ]:
import psutil
def get_size(bytes, suffix="B"):
    factor = 1024
    for unit in ["", "K", "M", "G", "T", "P"]:
        if bytes < factor:
            return f"{bytes:.2f}{unit}{suffix}"
        bytes /= factor
print("="*20, "Memory Information", "="*20)
svmem = psutil.virtual_memory()
print(f"Total: {get_size(svmem.total)}") ; print(f"Available: {get_size(svmem.available)}")
print(f"Used: {get_size(svmem.used)}") ; print(f"Percentage: {svmem.percent}%")

# GPU Information

In [ ]:
! nvidia-smi

# Preparing Colab

## Installing pkgs

In [ ]:
!apt install tree neofetch
!neofetch

In [ ]:
!pip install --upgrade pip
!pip install albumentations==0.5.1
!pip install opencv-python==4.4.0.46
import cv2
cv2.__version__

In [ ]:
!pip install tensorflow kaggle sklearn nltk --upgrade 
!pip install -U tensorboard_plugin_profile 
cv2.__version__

## Mount DRIVE

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
# dirs
KAGGLE = "drive/My Drive/NRB_project/kaggle.json"
drive_data_dir = "drive/My Drive/NRB_project/"
table_dir = "drive/My Drive/NRB_project/table_check.txt"

data_dir = "./data"
checks_dir = "./checkpoints"
csv_path = "./data/data.csv"

# loading_dir = './CRNN_model/'
# weights_dir = "drive/My Drive/NRB_project/CRNN"

loading_dir = './Multi_task_model'
weights_dir = "drive/My Drive/NRB_project/Multi_task"

In [ ]:
! mkdir ~/.kaggle
! mkdir ./data
! mkdir ./checkpoints

! cp "{table_dir}" "{data_dir}"

In [ ]:
!ls

In [ ]:
!ls "drive/My Drive/NRB_project/"

## Download Data

In [ ]:
! cp "{KAGGLE}" ~/.kaggle/
!kaggle datasets download -d tapakah68/yandextoloka-water-meters-dataset 

In [ ]:
! unzip -q yandextoloka-water-meters-dataset.zip -d data # unzipping
!mv data/WaterMeters/* data/                             # moving files to /data
!rm -r data/WaterMeters/                                 # removing empty folder

!ls "{data_dir}/"

# Imports

In [ ]:
import os
import cv2
import pprint
import random
import pandas as pd
import tensorflow as tf
# import tensorflow_addons as tfa

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
%matplotlib inline

import albumentations as A
import math
import time
import nltk

from nltk.translate.bleu_score import sentence_bleu
from nltk.translate.bleu_score import SmoothingFunction as smoothier

def asHHMMSS(s):
    m = math.floor(s / 60)
    s -= m * 60
    h = math.floor(m / 60)
    m -= h * 60
    return '%dh:%dm:%ds'% (h, m, s)

from collections import Counter
from sklearn.model_selection import train_test_split

print(f"Albumentations v: {A.__version__}")
print(f"TensorFlow v: {tf.__version__}")
print(f"Opencv v: {cv2.__version__}")

f"Timing: {asHHMMSS(162)}"

## Data paths

In [ ]:
SEED = 1234

IMAGES_path = os.path.join(data_dir, "images")
MASK_path = os.path.join(data_dir, "masks")
TABLE_path = os.path.join(data_dir, "table_check.txt") 

# Extra functions

In [ ]:
def set_seeds(seed=SEED):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    tf.random.set_seed(seed)
    np.random.seed(seed)

table = ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '.']
blank_idx = len(table)
def map2string(inputs):
    """Map outputs to number sequence."""
    strings = []
    if np.array(inputs).ndim > 1:
        for i in inputs:
            text = [table[char_index] for char_index in i 
                    if char_index != blank_idx]
            strings.append(''.join(text))
        return strings[0]
    else:
        text = [table[char] for char in inputs]
        return ''.join(text)

def load_from_csv(path, delimiter=','):
    """
    Load csv file and return a NumPy array of its data

    Parameters
    ----------
    path: str
        The path to the csv file to load
    delimiter: str (default: ',')
        The csv field delimiter

    Return
    ------
    D: array
        The NumPy array of the data contained in the file
    """
    if not os.path.exists(path):
        raise FileNotFoundError(f"File '{path}' does not exists.")
    return pd.read_csv(path, delimiter=delimiter, engine='python')
    # return pd.read_csv(path, delimiter=delimiter, float_precision='high', encoding='utf-8' )#, engine='python')
    # return pd.read_excel(path, header=None, names=['mails'])

def visualize(colormap='viridis', figsize=(16, 7), title=None, colorbar=None, **images):
    """PLot images in one row."""
    n = len(images)
    plt.figure(figsize=figsize)
    # plt.figure(figsize=(16, 10))
    for i, (name, image) in enumerate(images.items()):
        plt.subplot(1, n, i + 1)
        plt.axis("off")
        if isinstance(image, list):
            plt.title(' '.join(name.split('_')).title() +": " + detokenize(image[1]), fontsize=16)        
            plt.imshow(image[0], cmap=colormap)
        if title:
            plt.title(title, fontsize=42)        
            plt.imshow(image, cmap=colormap) 
        else:
            plt.title(' '.join(name.split('_')).title(), fontsize=16)        
            plt.imshow(image, cmap=colormap)        
        if colorbar:
            plt.colorbar()
    plt.show()

# helper function for data visualization    
def denormalize(x, BGR_RGB=True):
    """Scale image to range 0..1 for correct plot"""
    x_max = np.percentile(x, 98)
    x_min = np.percentile(x, 2)    
    x = (x - x_min) / (x_max - x_min)
    x = x.clip(0, 1) 
    if BGR_RGB:
        x = cv2.cvtColor(x, cv2.COLOR_BGR2RGB)
    return x
def normalize_0_255(img):
    '''Return array with values in range [0, 255]'''    
    norm_image = cv2.normalize(np.float32(img), None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_32F)
    return norm_image.astype(np.uint8)

def crop_with_argwhere(image, mask):
    # https://codereview.stackexchange.com/questions/132914/crop-black-border-of-image-using-numpy
    # mask should has values > 0
    # Coordinates of non-black pixels.
    coords = np.argwhere(mask)
    # Bounding box of non-black pixels.
    x0, y0 = coords.min(axis=0)
    x1, y1 = coords.max(axis=0) + 1   # slices are exclusive at the top
    # Get the contents of the bounding box.
    cropped = image[x0:x1, y0:y1]
    return cropped

def create_crop_save_images(imgs_path, folder, color="RGB", kernel_sz=20
                            , apply_augmentation=False):
    '''Read, crop and save images'''
    images_fps = [os.path.join(IMAGES_path, image_id) for image_id in imgs_path]
    masks_fps = [os.path.join(MASK_path, image_id) for image_id in imgs_path] 
    to_dilate = True if kernel_sz else False

    folder2save = os.path.join(data_dir, folder)
    if not os.path.exists(folder2save):
        os.makedirs(folder2save)
    final_path = []
    counter = 0
    for i in range(len(images_fps)):
        # change extension from jpg to png 
        path2save = os.path.join(folder2save, imgs_path[i].rsplit('.',1)[0]+'.png')
        if path2save in final_path: 
            # do not overwrite images when data augment            
            path2save = path2save.replace("_v", "-2_v")
            counter +=1
        if color=="RGB":
            image = cv2.imread(images_fps[i])
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        elif color=="gray": # gray scale
            image = cv2.imread(images_fps[i], 0)   

        mask = cv2.imread(masks_fps[i], 0) # 0 = cv2.COLOR_GRAYSCALE  
        # if apply_augmentation: # here can also be include the MASK
        #     sample = apply_augmentation(image=image)
        #     image = sample['image']
                        
        image=cv2.bitwise_and(image, image, mask=mask)  # mask selection
        
        if to_dilate:
            kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (kernel_sz, kernel_sz))
            dilate = cv2.dilate(mask, kernel, iterations=3)
            image= crop_with_argwhere(image, dilate) # cropping to dilated size
            mask= crop_with_argwhere(mask, dilate)
        else:
            image = crop_with_argwhere(image, mask) # cropping to mask size
            mask = crop_with_argwhere(mask, mask)
        # apply alignment with the given mask
        # image = align_rotate(image=image, with_mask=mask)
        image = align_rotate(inputs=[image, mask] )
        if apply_augmentation: # here can also be include the MASK
            sample = apply_augmentation(image=image)
            image = sample['image']

        final_path.append(path2save)
        cv2.imwrite(path2save, cv2.cvtColor(image, cv2.COLOR_RGB2BGR))
    if counter: print("Repetitions:", counter)
    return final_path
# !rm -r data/train
# create_crop_save_images(X_valid[:3], "None", kernel_sz=None, apply_augmentation=get_training_augmentation())

In [ ]:
# taken from here: https://stackoverflow.com/questions/56459002/how-to-detect-and-align-tilted-images-after-cropping
def order_corner_points(corners):
    # Separate corners into individual points
    # Index 0 - top-right
    #       1 - top-left
    #       2 - bottom-left
    #       3 - bottom-right
    if len(corners) == 4:
        top_l, top_r, bottom_r, bottom_l = corners[2], corners[3], corners[0], \
                                           corners[1]
        return (top_l, top_r, bottom_r, bottom_l)
    corners = [(corner[0][0], corner[0][1]) for corner in corners]
    top_r, top_l, bottom_l, bottom_r = corners[0], corners[1], corners[2], \
                                       corners[3]
    return (top_l, top_r, bottom_r, bottom_l)

def perspective_transform(image, corners):
    # Order points in clockwise order
    ordered_corners = order_corner_points(corners)
    top_l, top_r, bottom_r, bottom_l = ordered_corners

    # Determine width of new image which is the max distance between
    # (bottom right and bottom left) or (top right and top left) x-coordinates
    width_A = np.sqrt(((bottom_r[0] - bottom_l[0]) ** 2) + (
                (bottom_r[1] - bottom_l[1]) ** 2))
    width_B = np.sqrt(
        ((top_r[0] - top_l[0]) ** 2) + ((top_r[1] - top_l[1]) ** 2))
    width = max(int(width_A), int(width_B))

    # Determine height of new image which is the max distance between
    # (top right and bottom right) or (top left and bottom left) y-coordinates
    height_A = np.sqrt(
        ((top_r[0] - bottom_r[0]) ** 2) + ((top_r[1] - bottom_r[1]) ** 2))
    height_B = np.sqrt(
        ((top_l[0] - bottom_l[0]) ** 2) + ((top_l[1] - bottom_l[1]) ** 2))
    height = max(int(height_A), int(height_B))

    # Construct new points to obtain top-down view of image in
    # top_r, top_l, bottom_l, bottom_r order
    dimensions = np.array([[0, 0], [width - 1, 0], [width - 1, height - 1],
                           [0, height - 1]], dtype="float32")

    # Convert to Numpy format
    ordered_corners = np.array(ordered_corners, dtype="float32")

    # Find perspective transform matrix
    matrix = cv2.getPerspectiveTransform(ordered_corners, dimensions)

    # Return the transformed image
    return cv2.warpPerspective(image, matrix, (width, height))

def get_image_width_height(image):
    image_width = image.shape[1]  # current image's width
    image_height = image.shape[0]  # current image's height
    return image_width, image_height

def calculate_scaled_dimension(scale, image):
    image_width, image_height = get_image_width_height(image)
    ratio_of_new_with_to_old = scale / image_width

    dimension = (scale, int(image_height * ratio_of_new_with_to_old))
    return dimension

def scale_image(image, size):
    image_resized_scaled = cv2.resize(
        image,
        calculate_scaled_dimension(
            size,
            image
        ),
        interpolation=cv2.INTER_AREA
    )
    return image_resized_scaled

def rotate_image(image, angle):
    # Grab the dimensions of the image and then determine the center
    (h, w) = image.shape[:2]
    (cX, cY) = (w / 2, h / 2)

    # grab the rotation matrix (applying the negative of the
    # angle to rotate clockwise), then grab the sine and cosine
    # (i.e., the rotation components of the matrix)
    M = cv2.getRotationMatrix2D((cX, cY), -angle, 1.0)
    cos = np.abs(M[0, 0])
    sin = np.abs(M[0, 1])

    # Compute the new bounding dimensions of the image
    nW = int((h * sin) + (w * cos))
    nH = int((h * cos) + (w * sin))

    # Adjust the rotation matrix to take into account translation
    M[0, 2] += (nW / 2) - cX
    M[1, 2] += (nH / 2) - cY

    # Perform the actual rotation and return the image
    return cv2.warpAffine(image, M, (nW, nH))

def align_rotate(inputs, to_plot=False, to_save=False, to_scale:int=None
                      , to_rotate=True, path2save=None):
    
    # if len(inputs) == 2:
    #     # copy original image to plot later if neded
    #     image, mask = inputs
    #     original_image = image.copy()        
    #     # mask = cv2.imread(with_mask, 0)
    # else:
    #     original_image = image.copy()
    # if to_scale:
    #     image = scale_image(image, to_scale)
    #     mask = scale_image(mask, to_scale) if len(inputs)==2 else None
    
    if len(inputs) != 2:
        original_image = inputs.copy()
        if to_scale:  # if you want to enlarge the image
            image = scale_image(inputs, to_scale)
        # convert the image to grayscale, blur it, and find edges in the image
        gray_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        # gray_rgb = cv2.bilateralFilter(gray_rgb, 11, 17, 17)
        # gray_rgb = cv2.medianBlur(gray_rgb, 7) # working
        # gray_rgb = cv2.medianBlur(gray_rgb, 9) # working
        gray_rgb = cv2.medianBlur(gray_rgb, 5) # working too
        edged = cv2.Canny(gray_rgb, 30, 200)  # edges
        # edged = cv2.Canny(gray_rgb, 30, 200, L2gradient=True, apertureSize=3)  # edges
    else:
        image, mask = inputs
        original_image = image.copy()   
        if to_scale: # if you want to enlarge the image
            image = scale_image(image, to_scale)
            mask = scale_image(mask, to_scale)
        mask = cv2.medianBlur(mask, 5)
        edged = cv2.Canny(mask, 30, 200)
    # search for contours | returns contours and hierarchy of them
    cnts = cv2.findContours(edged.copy(), cv2.RETR_TREE,
                            cv2.CHAIN_APPROX_SIMPLE)
    cnts = cnts[0] if len(cnts) == 2 else cnts[1]  # drops the hierarchy
    # sort them by area from higher to lower size | picks the top10
    cnts = sorted(cnts, key=cv2.contourArea, reverse=True)[:10]

    img_contours = image.copy()
    # loop through the contours and apply perspective transform
    for idx in range(len(cnts)):
        cv2.drawContours(img_contours, cnts, idx, (0, 255, 0), 1)
        # find minimal rectangle for the contour
        rect = cv2.minAreaRect(cnts[idx])
        box = cv2.boxPoints(rect)
        box = np.int0(box)
        # change perspective of the rectangle in the image
        transformed = perspective_transform(image, box)
        break

    if to_rotate:
        # rotate if necessary
        (h, w) = transformed.shape[:2]        
        if (h > w):
            rotated = rotate_image(transformed, 90)
        else:
            rotated = transformed

        return rotated
    if to_plot:
        cv2.namedWindow("image", cv2.WINDOW_NORMAL)
        cv2.namedWindow("transformed", cv2.WINDOW_NORMAL)
        cv2.namedWindow("contours", cv2.WINDOW_NORMAL)
        if to_rotate:
            cv2.namedWindow("rotated", cv2.WINDOW_NORMAL)
            cv2.imshow("rotated", rotated)
        cv2.imshow("transformed", transformed)
        cv2.imshow("image", original_image)
        cv2.imshow("contours", img_contours)
        cv2.waitKey(0)
    if to_save:
        cv2.imwrite(os.path.join("aligned", path2save), rotated)
    return transformed

## Augmentation & Preprocessing

In [ ]:
# img = "data/train/id_1070_value_102_828.png"
# img = cv2.imread(img)
# img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# img2 = "data/train/id_1070-2_value_102_828.png"
# img2 = cv2.imread(img2)
# img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)


transform = get_training_augmentation()

for _ in range(5):
    for i in X_valid[20:30]:
        img = os.path.join('data/valid/', i)
        img = img.replace(".jpg", '.png')
        img = cv2.imread(img)        
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        transformed = transform(image=img)
        transformed = transformed['image']
        # gray = tf.keras.preprocessing.image.array_to_img(transformed)
        gray = tf.image.rgb_to_grayscale(transformed)
        visualize(
            image=img,
            transformed=transformed,
            # gray=gray[..., 0], colormap='gray',
            figsize=(20, 10)
        )
        print(img.shape, transformed.shape, np.shape(gray))

In [ ]:
def round_clip_0_1(x, **kwargs):
    return x.round().clip(0, 1)

def get_training_augmentation():
    train_transform = [
        
        # A.ShiftScaleRotate(p=.5, border_mode=0, 
        #                     rotate_limit=15,),

        # A.PadIfNeeded(min_height=320, min_width=320, 
        #               always_apply=True, border_mode=0),

        # A.CenterCrop (height=320, width=320, always_apply=True),
        # A.RandomCrop(height=320, width=320, always_apply=True),
        # A.CropNonEmptyMaskIfExists (height=320, width=320, always_apply=True),
         

        A.OneOf( # problem `HueSaturationValue` does not work for gray images
            [
                A.RandomContrast(p=1),
                A.HueSaturationValue(p=1),
            ],
            p=0.667,
            # # p=0.7667,
            # p=1,
        ),

        A.OneOf(
            [
                A.CLAHE(p=1),
                A.RandomBrightness(p=1, limit=(-0.1, 0.3)),
                # A.RandomGamma(p=1),
            ],
            # p=0.667,
            p=0.7667,
            # p=1,
        ),

        A.OneOf(
            [
                A.IAASharpen(p=1),
                A.Blur(blur_limit=5, p=1),
                A.MotionBlur(blur_limit=5, p=1),
            ],
            # # p=0.667,
            p=0.7667,
            # p=1,
        ),
        # # ------------------
        A.OneOf(
            [
        A.MultiplicativeNoise(multiplier=[.5, 1.5], elementwise=False, p=1),
        A.JpegCompression(quality_lower=26, quality_upper=27, p=1),
            ],
            p=0.667
            # p=1
        ),        
        # A.Cutout(num_holes=20, max_h_size=5, max_w_size=5, fill_value=0, p=0.33), # does not work - images have diff sizes
        # ------------------
        A.Rotate(limit=5, p=0.33, border_mode=0),
        # A.Rotate(limit=5, p=0.37, border_mode=0),
        A.Lambda(mask=round_clip_0_1),
    ]
    return A.Compose(train_transform)

def preprocess_data_input(image=None, mask=None, **kwargs):
    """Preprocess Images according to `ImageNet`
    Args:
        image: image to process
        mask: mask of the image
        kwargs: inserted to not break the convention of 
                albumentations.Compose
    Return:
        function from keras that process the image.    
    """
    return tf.keras.applications.imagenet_utils.preprocess_input(image.copy(), mode="caffe")

def get_validation_augmentation():
    """Add paddings to make image shape divisible by 32"""
    test_transform = [
        # A.PadIfNeeded(384, 480)
        # A.PadIfNeeded(min_height=320, min_width=320, 
        #               always_apply=True, border_mode=0),
        A.Rotate(limit=5, p=1.0)
    ]
    return A.Compose(test_transform)

def get_preprocessing(preprocessing_fn):
    """Construct preprocessing transform
    
    Args:
        preprocessing_fn (callbale): data normalization function 
            (can be specific for each pretrained neural network)
    Return:
        transform: albumentations.Compose
    
    """    
    _transform = [
        A.Lambda(image=preprocessing_fn),
    ]
    return A.Compose(_transform)

# DATA

In [ ]:
def grab_value_from_name(sentence, sep="."):
    '''Function to grab the actual value from image name; due to problems in the CSV file.'''
    start, end = "value_", ".jpg"
    return sentence[sentence.find(start)+len(start) : sentence.rfind(end)].replace("_", sep)

def plot_occurrences(values, per_posi=None, remove=None):
    vals = [v for value in values for v in value]
    vocab_count = Counter(vals)

    # fig = plt.figure(figsize = (13, 7)) 
    fig = plt.figure(figsize = (15, 9)) 
    ctr = 0
    tab_vals = ['.', '0', '1','2','3','4','5','6','7','8','9']
    if remove: tab_vals.remove(remove)
    for i in tab_vals:
        if per_posi is not None:
            previous = 0
            for key, val in per_posi[i].items():
                col = cm.nipy_spectral( float(key) / 9)
                plt.bar(i, val, bottom=previous, color=col)
                previous= previous + val
        else:
            plt.bar(i, vocab_count[i], color='lightblue')
        plt.annotate(f"{vocab_count[i]}",
                     xy= (ctr , vocab_count[i] + 8),
                     ha='center', va='bottom', fontsize=12
        )
        ctr+=1

    plt.grid(alpha=.4)
    # plt.title("Ocurrences per token",  fontsize=18)
    plt.ylabel("Number of occurrences", fontsize=14)
    plt.xlabel("Tokens", fontsize=14)
    plt.xticks(fontsize=14)
    plt.yticks(fontsize=14)
    if per_posi is not None:
        plt.legend([f'{n}-position' for n in range(1,10)], fontsize=14)

def get_perPosition(labels):
    per_position = {}.fromkeys(['.', '0', '1','2','3','4','5','6','7','8','9'], {}.fromkeys([str(i) for i in range( 9)], 0))
    per_position = pd.DataFrame.from_dict(per_position)
    for label in labels:
        for idx in range(len(label)):
            per_position[label[idx]][str(idx)] += 1
    return per_position

def pad_labels(labels, sep="."):
    padded_labels = []
    # max_l = max(len(lbl) for lbl in labels)
    max_left = max(len(lbl.split(sep)[0] ) for lbl in labels)
    max_right = max(len(lbl.split(sep)[1]) for lbl in labels)
    ctr_left, ctr_lo, ctr_sht = 0,0,0
    sh_r_nbs, lo_r_nbs = [], []
    for lab in labels:
        _fill_left = max_left - len(lab.split(sep)[0])
        if _fill_left > 0: # count left side of the sequence
            ctr_left+=1
        _fill_right = max_right - len(lab.split(sep)[1])
        if _fill_right < 2 and _fill_right is not 0: # filling less 0s means long seqs
            lo_r_nbs.append((labels.index(lab), lab))
            ctr_lo+=1
        elif _fill_right > 1: # short sequences
            sh_r_nbs.append((labels.index(lab), lab))
            ctr_sht+=1
        lab = _fill_left * "0" + lab
        lab = lab + "0"*_fill_right
        padded_labels.append(lab)
    print(f"Padded counts; left: {ctr_left}, short-right: {ctr_sht} long-right: {ctr_lo}")
    return padded_labels, sh_r_nbs, lo_r_nbs

CSV = load_from_csv(csv_path)
print(CSV.head(10))

## Brief analysis of the data images

In [ ]:
X = CSV['photo_name'].tolist()
X = np.array(X)

labels_noPad = [grab_value_from_name(name, ".") for name in X] # raw labels

In [ ]:
newPosition = get_perPosition(labels_noPad)
plot_occurrences(labels_noPad, newPosition)

In [ ]:
X = CSV['photo_name'].tolist()
X.remove("id_664_value_45_2676.jpg") # remove image with the longest floats
X = np.array(X)

labels_noPad = [grab_value_from_name(name, ".") for name in X] # raw labels
labels, short_r_seqs, long_r_seqs = pad_labels(labels_noPad)

print(f'Right-padded sequences: short: {len(short_r_seqs)}, long: {len(long_r_seqs)}')
# grabbing images that start with 0 from the short_seqs
all_short_seqs = [X[id] for (id, seq) in short_r_seqs if seq.rsplit(".", 1)[1].startswith('0')]
# remove 3 images that actually are large sequences
for i in ['id_799_value_22_0.jpg', 'id_563_value_3_0.jpg', 'id_478_value_226_0.jpg']:
    all_short_seqs.remove(i) 
print(f'Total nb of images to remove from data: {len(all_short_seqs)}')

# start over, but removing short images
X = CSV['photo_name'].tolist()
# remove list of short sequences
before = len(X)
X.remove("id_664_value_45_2676.jpg") # remove image with the longest floats
for id in all_short_seqs:
    X.remove(id)
print(f"\nData length --> before: {before}, after: {len(X)}")
X = np.array(X)

labels_noPad = [grab_value_from_name(name, ".") for name in X] # raw labels
labels, _, _ = pad_labels(labels_noPad)

print("\nFinal data shape:", X.shape, np.array(labels).shape)

In [ ]:
newPosition = get_perPosition(labels)
print(newPosition)
plot_occurrences(labels, newPosition)

## Data for CNNs

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(X, np.array(labels), test_size=0.3, random_state=SEED)
print("New Train-Test shape:", X_train.shape, X_valid.shape, y_train.shape, y_valid.shape)
# New Train-Test shape: (870,) (373,) (870,) (373,)

- In the following cells, extra samples from the same training set were added to the training set.
- Preliminary results show no importance improvements in the model performance

In [ ]:
rand_samples = np.random.RandomState(SEED).choice(X_train, size=int(len(X_train)*.33), replace=0)
rand_targets = np.random.RandomState(SEED).choice(y_train, size=int(len(X_train)*.33), replace=False)

len(rand_samples), len(rand_targets), len(np.unique(rand_samples)), len(np.unique(rand_targets))

In [ ]:
X_train = np.concatenate( (X_train, rand_samples) )
y_train = np.concatenate( (y_train, rand_targets) )

print(f"New Train shape: {X_train.shape}, {y_train.shape}")
print(f"Unique images: {len(np.unique(X_train))}")

In [ ]:
for i, j in zip(X_train[844:865], y_train[844:865]):
    print(i,'\t\t',j)

### Pre-processing images

- Crop, alignment and augmentation

In [ ]:
X_cut_train = create_crop_save_images(X_train, "train", kernel_sz=11, apply_augmentation=get_training_augmentation())
# X_cut_train = create_crop_save_images(X_train, "train", kernel_sz=11)
X_cut_valid = create_crop_save_images(X_valid, "valid", kernel_sz=11)

print(f'Train set: {len(X_cut_train)} -- Valid set: {len(X_cut_valid)}')

In [ ]:
# count files in data/train folder
!ls data/train/ | wc -l

In [ ]:
img = "data/train/id_1070_value_102_828.png"
img = cv2.imread(img)
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

# img2 = "data/train/id_1070-2_value_102_828.png"
# img2 = cv2.imread(img2)
# img2 = cv2.cvtColor(img2, cv2.COLOR_BGR2RGB)

visualize(
    image=img,
    # image2=img2,
)

In [ ]:
# !rm -r ./data/train/
# !rm -r ./data/valid/

# MODEL CNNs : Multi-task
- Convolutional Nets

In [ ]:
from tensorflow.keras.layers import *
from tensorflow.keras.regularizers import L2

tf.keras.backend.clear_session()

def build_model(nb_classes, shape=(64, 256, 1), weight_decay=0, batch_momentum=0.99):
    ACTIV="swish"
    # ACTIV='relu'
    bn_axis = 3
    input_ = tf.keras.layers.Input(shape=shape)
    x = Conv2D(128, (3, 3), padding='same', use_bias=False, name='block1_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay) )(input_)
    x = BatchNormalization(axis=bn_axis, name='bn00_x1', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)

    x = Conv2D(128, (3, 3), padding='same', use_bias=False, name='block1_conv2', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn01_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block1_pool')(x)

    # # Block 2
    x = Conv2D(128, (3, 3), padding='same', use_bias=False, name='block2_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn11_x1', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)

    x = Conv2D(128, (3, 3), padding='same', use_bias=False, name='block2_conv2', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn12_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block2_pool')(x)

    # # Block 3
    x = Conv2D(256, (3, 3), padding='same', use_bias=False, name='block3_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn21_x1', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)

    x = Conv2D(256, (3, 3), padding='same', use_bias=False, name='block3_conv2', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn22_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block3_pool')(x)

    # Block 4
    x = Conv2D(512, (3, 3), padding='same', use_bias=False, name='block4_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn31_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    
    x = Conv2D(512, (3, 3), padding='same', use_bias=False, name='block4_conv2', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn32_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block4_pool')(x)
    
    # Block 5
    x = Conv2D(512, (3, 3), padding='same', use_bias=False, name='block5_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn41_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    
    x = Conv2D(1024, (3, 3), padding='same', use_bias=False, name='block5_conv3', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn43_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block5_pool')(x)
    
    # Block 6
    x = Conv2D(1024, (3, 3), padding='same', use_bias=False, name='block6_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn51_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    
    x = Conv2D(1024*2, (3, 3), padding='same', use_bias=False, name='block6_conv12', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn51_x22', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    x = tf.keras.layers.Dropout(0.3)(x)


    # Max-pooling layers are worse at preserving localization --> AveragePooling fits here better
    X = AveragePooling2D((2, 2), strides=(2, 1), name='avg_pool1', padding ='same')(x)
    
    X = tf.keras.layers.Reshape((-1, 1024*2), name='Reshape')(X)
    # block7     
    X = Conv1D(512, 3, strides=1, padding='same', name = 'conv1y', activation=None,  use_bias=False, kernel_initializer="he_normal", kernel_regularizer=L2(0.000))(X)
    X = BatchNormalization(axis=2, name = 'bn01y')(X)
    X = Activation(ACTIV)(X)
    X = tf.keras.layers.Dropout(0.3)(X)
    X = tf.keras.layers.Conv1D(filters=nb_classes, kernel_size=1 , strides=1, padding='same', name='conv1x', activation=None,  use_bias=False, kernel_initializer="he_normal")(X)
    X = tf.keras.layers.BatchNormalization(axis=2, name='bnhe')(X)
    X = tf.keras.layers.Activation('softmax', name='Softmax')(X)

    model = tf.keras.Model(inputs=input_, outputs=X, name='Multi_task')
    return model
    # bn_axis = 3
    # input_ = tf.keras.layers.Input(shape=shape)
    # x = Conv2D(128, (3, 3), padding='same', use_bias=False, name='block1_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay) )(input_)
    # x = BatchNormalization(axis=bn_axis, name='bn00_x1', momentum=batch_momentum)(x)
    # x = Activation(ACTIV)(x)
    # x = Conv2D(128, (3, 3), padding='same', use_bias=False, name='block1_conv2', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    # x = BatchNormalization(axis=bn_axis, name='bn01_x2', momentum=batch_momentum)(x)
    # x = Activation(ACTIV)(x)
    # x = MaxPooling2D((2, 2), strides=(2, 2), name='block1_pool')(x)

    # # # Block 2
    # x = Conv2D(128, (3, 3), padding='same', use_bias=False, name='block2_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    # x = BatchNormalization(axis=bn_axis, name='bn11_x1', momentum=batch_momentum)(x)
    # x = Activation(ACTIV)(x)

    # x = Conv2D(128, (3, 3), padding='same', use_bias=False, name='block2_conv2', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    # x = BatchNormalization(axis=bn_axis, name='bn12_x2', momentum=batch_momentum)(x)
    # x = Activation(ACTIV)(x)
    # x = MaxPooling2D((2, 2), strides=(2, 2), name='block2_pool')(x)

    # # # Block 3
    # x = Conv2D(256, (3, 3), padding='same', use_bias=False, name='block3_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    # x = BatchNormalization(axis=bn_axis, name='bn21_x1', momentum=batch_momentum)(x)
    # x = Activation(ACTIV)(x)

    # x = Conv2D(256, (3, 3), padding='same', use_bias=False, name='block3_conv2', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    # x = BatchNormalization(axis=bn_axis, name='bn22_x2', momentum=batch_momentum)(x)
    # x = Activation(ACTIV)(x)
    # x = MaxPooling2D((2, 2), strides=(2, 2), name='block3_pool')(x)

    # # Block 4
    # x = Conv2D(512, (3, 3), padding='same', use_bias=False, name='block4_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    # x = BatchNormalization(axis=bn_axis, name='bn31_x2', momentum=batch_momentum)(x)
    # x = Activation(ACTIV)(x)
    
    # x = Conv2D(512, (3, 3), padding='same', use_bias=False, name='block4_conv2', kernel_regularizer=L2(weight_decay))(x)
    # x = BatchNormalization(axis=bn_axis, name='bn32_x2', momentum=batch_momentum)(x)
    # x = Activation(ACTIV)(x)
    # x = MaxPooling2D((2, 2), strides=(2, 2), name='block4_pool')(x)

    # # Block 5
    # x = Conv2D(512, (3, 3), padding='same', use_bias=False, name='block5_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    # x = BatchNormalization(axis=bn_axis, name='bn41_x2', momentum=batch_momentum)(x)
    # x = Activation(ACTIV)(x)
    
    # x = Conv2D(1024, (3, 3), padding='same', use_bias=False, name='block5_conv3', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    # x = BatchNormalization(axis=bn_axis, name='bn43_x2', momentum=batch_momentum)(x)
    # x = Activation(ACTIV)(x)
    # x = MaxPooling2D((2, 2), strides=(2, 2), name='block5_pool')(x)
    
    # x = Conv2D(1024, (3, 3), padding='same', use_bias=False, name='block6_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    # x = BatchNormalization(axis=bn_axis, name='bn51_x2', momentum=batch_momentum)(x)
    # x = Activation(ACTIV)(x)
    
    # x = Conv2D(1024*2, (3, 3), padding='same', use_bias=False, name='block6_conv12', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    # x = BatchNormalization(axis=bn_axis, name='bn51_x22', momentum=batch_momentum)(x)
    # x = Activation(ACTIV)(x)
    # x = tf.keras.layers.Dropout(0.3)(x)
   
    # # Max-pooling layers are worse at preserving localization --> AveragePooling fits here better
    # X = AveragePooling2D((2, 2), strides = (2, 1), name='avg_pool1', padding ='same')(x)
    
    # # block6 
    # X = tf.keras.layers.Reshape((-1, 1024*2), name='Reshape')(X)
        
    # X = Conv1D(512, 3, strides=1, padding='same', name = 'conv1y', activation=None,  dilation_rate=1,  use_bias=False, kernel_initializer="he_normal", kernel_regularizer=L2(0.000))(X)
    # X = BatchNormalization(axis=2, name = 'bn01y')(X)
    # X = Activation(ACTIV)(X)
    # X = tf.keras.layers.Dropout(0.3)(X) # weird use_bias=True -> should be False
    # X = tf.keras.layers.Conv1D(filters=nb_classes, kernel_size=1 , strides=1, padding='same', name = 'conv1x', activation=None,  dilation_rate=1,  use_bias=True, kernel_initializer="he_normal")(X)
    # X = tf.keras.layers.BatchNormalization(axis = 2, name = 'bnhe')(X)
    # X = tf.keras.layers.Activation('softmax', name='Softmax')(X)

    # model = tf.keras.Model(inputs=input_, outputs=X, name='Multi_task')
    # return model

# Hyper-parameters

In [ ]:
BATCH_SIZE = 16 #12 # 32 # best 16
EPOCHS = 50 # 80
CLIP_NORM = 1.0 # or 1.0 # or 5.0
IMG_CHANNELS = 3 # 3 # 1 to GRAY OR 3 to RGB
IMG_HEIGHT = 64 
IMG_WIDTH = 288
CLIP_VALUE = 5.0


# Training process

### Watermeter dataset (russian)

In [ ]:
class DatasetBuilder(): # for images of the RUSSIAN dataset

    def __init__(self, table_path, img_channels, max_img_width=320,
                 img_width=None, img_height=32):
        self.table = tf.lookup.StaticHashTable(tf.lookup.TextFileInitializer(
            table_path, tf.string, tf.lookup.TextFileIndex.WHOLE_LINE, 
            tf.int64, tf.lookup.TextFileIndex.LINE_NUMBER,), default_value=-1)
        self.img_height = img_height
        self.img_channels = img_channels
        if img_width is None:
            self.max_img_width = max_img_width
            self.preserve_aspect_ratio = True
        else:
            self.img_width = img_width
            self.preserve_aspect_ratio = False
    @property
    def num_classes(self):
        return self.table.size()

    def _decode_img(self, filename, label):
        img = tf.io.read_file(filename)
        img = tf.io.decode_png(img, channels=self.img_channels)
        if self.preserve_aspect_ratio:
            img_shape = tf.shape(img)
            scale_factor = self.img_height / img_shape[0]
            img_width = scale_factor * tf.cast(img_shape[1], tf.float64)
            img_width = tf.cast(img_width, tf.int32)
        else:
            img_width = self.img_width
        
        # img = tf.image.resize(img, (self.img_height, img_width)) / 255.0
        img = tf.image.resize(img, (self.img_height, img_width)) 
        # by dividing to 255.0 is a way to normalize between 0-1
        if self.img_channels == 3:      
            img = tf.keras.applications.imagenet_utils.preprocess_input(img, mode="caffe")
        return img, label

    def _filter_img(self, img, label):
        """ Keep images that have width < max_img_width. """
        img_shape = tf.shape(img)
        return img_shape[1] <= self.max_img_width

    def _tokenize(self, imgs, label):
        chars = tf.strings.unicode_split(label, 'UTF-8')
        tokens = tf.ragged.map_flat_values(self.table.lookup, chars)
        # tokens = tokens.to_sparse()
        tokens = tf.keras.utils.to_categorical(tokens, self.table.size())

        return imgs, tokens
        
    def build(self, img_paths, labels, batch_size, is_training):
        """
        build dataset: read image and format labels.
        """
        chars = tf.strings.unicode_split(labels, 'UTF-8')
        tokens = tf.ragged.map_flat_values(self.table.lookup, chars)
        labels_categorized = [tf.keras.utils.to_categorical(
            token, self.table.size()) for token in tokens]
            
        ds = tf.data.Dataset.from_tensor_slices((img_paths, labels_categorized))
        if is_training:
            ds = ds.shuffle(buffer_size=10000)
        ds = ds.map(self._decode_img,  # reshape images inside data_builder
                    num_parallel_calls=tf.data.experimental.AUTOTUNE)
        # Ignore the errors e.g. decode error or invalid data.
        # ds = ds.apply(tf.data.experimental.ignore_errors())
        if self.preserve_aspect_ratio and batch_size != 1:
            ds = ds.filter(self._filter_img)
            ds = ds.padded_batch(batch_size, drop_remainder=is_training)
        else:
            ds = ds.batch(batch_size, drop_remainder=is_training)
        ds = ds.prefetch(tf.data.experimental.AUTOTUNE)
        return ds        
        

In [ ]:
X_cut_valid = [os.path.join("./data/valid", x.rsplit('.', 1)[0]+".png") for x in X_valid]
X_cut_train = [os.path.join("./data/train", x.rsplit('.', 1)[0]+".png") for x in X_train]
# uniques = []
# counter =0
# for i in range(len(X_cut_train)):
#     if X_cut_train[i] in uniques:
#         X_cut_train[i] = X_cut_train[i].replace("_v", "-2_v")
#         counter +=1
#     uniques.append(X_cut_train[i])
len(X_cut_train), len(X_cut_valid)

In [ ]:
X_cut_valid[55], X_valid[55], y_valid[55]

In [ ]:
data_builder = DatasetBuilder(TABLE_path, img_channels=IMG_CHANNELS, img_height=IMG_HEIGHT
                              , img_width=IMG_WIDTH
                              )
data_builder.num_classes
train_loader = data_builder.build(X_cut_train, y_train, BATCH_SIZE, is_training=True)
val_loader = data_builder.build(X_cut_valid, y_valid, BATCH_SIZE, False)

In [ ]:
for (batch, (inp, tar)) in enumerate(val_loader):
# for (batch, (inp, tar)) in enumerate(train_loader):
    print(batch, inp.shape, tar.shape)
    break


In [ ]:
for i in range(len(inp)):
    visualize(        
        # img = inp[i][..., 0]
        # img = inp[i]
        img = denormalize( inp[i].numpy() )
        # , colormap="gray"
    )
    print("image:", i ,inp[i].dtype, inp[i].shape, np.max(inp[i]), np.min(inp[i]))
    print(f"target: { map2string( np.argmax(tar[i], axis=1) ) }")  

## Build model

In [ ]:
# optimizer = tf.keras.optimizers.Adam(0.0005, clipnorm=CLIP_NORM) # SO FAR THE BEST
optimizer = tf.keras.optimizers.Adam(0.00058, clipnorm=CLIP_NORM) # SO FAR THE BEST


# optimizer = tf.keras.optimizers.Adam(0.0002, clipnorm=CLIP_NORM) # 
# optimizer = tf.keras.optimizers.Adam(0.0002) # SO FAR working
# optimizer = tf.keras.optimizers.SGD(learning_rate=0.0002, momentum=0.9, clipnorm=CLIP_NORM)
# optimizer = tf.keras.optimizers.RMSprop(0.0002, clipnorm=CLIP_NORM)
# optimizer = tf.keras.optimizers.RMSprop(0.0002, clipvalue=CLIP_VALUE)

optimizer.learning_rate

In [ ]:
model = build_model(nb_classes=data_builder.num_classes.numpy(), shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)) 

## Loss and Accuracy

In [ ]:
def loss_funct(y_true, y_pred):
    shapes = tf.shape(y_pred)
    y_true = tf.reshape(y_true, (-1, shapes[-1]))
    y_pred = tf.reshape(y_pred, (-1, shapes[-1]))
    # loss = tf.keras.backend.sum(
    loss = tf.reduce_sum(
        tf.keras.losses.categorical_crossentropy(y_true, y_pred))
    
    batch_size = tf.shape(y_true)[0]
    batch_size = tf.cast(batch_size, tf.float32)
       
    return tf.reduce_mean(loss)/batch_size
    # return tf.keras.backend.mean(loss)/batch_size


def edit_dist(y_true, y_pred):
    '''Calculates the Levenshtein distance between sequences.'''

    y_pred_shape = tf.shape(y_pred)
    batch_size = y_pred_shape[0]
    batch_size = tf.cast(batch_size, tf.float32)
    
    y_pred = tf.argmax(y_pred, axis=-1)
    y_true = tf.argmax(y_true, axis=-1)
    y_pred = tf.sparse.from_dense(y_pred)
    y_true = tf.sparse.from_dense(y_true)

    sum_distance = tf.math.reduce_sum(tf.edit_distance(y_pred, y_true))
    return sum_distance/batch_size

def seq_accuracy(y_true, y_pred):
    ''' Calculates the sequence accuracy between 
        the true and predicted sequence. 
    '''
	# correctly classified
    y_pred =  tf.argmax(y_pred, axis=-1)
    y_true = tf.argmax(y_true, axis=-1)

    correct_number_per_seq = tf.cast(tf.reduce_all( tf.equal(y_pred, y_true), axis=-1), dtype=tf.float32)

    batch_size = tf.shape(y_true)[0]
    batch_size = tf.cast(batch_size, tf.float32)

    # return tf.keras.backend.sum(correct_number_per_seq) / batch_size
    return tf.reduce_sum(correct_number_per_seq) / batch_size

## Callbacks

In [ ]:
# delete previous session
!rm -r logs_Multi_task/
!rm -r Multi_task_model/ 


dir_name = "./{}_{}".format(model.name, 'model')
print(f"Model checkpoints dir: {dir_name}")
!mkdir $dir_name

callbacks = [

    tf.keras.callbacks.ModelCheckpoint(
                '{}/{}_{}_best_model.h5'.format(dir_name, model.name,
                                        'model')
                    , save_weights_only=True, save_best_only=True, mode='max'
                    , monitor='val_seq_accuracy'),
    tf.keras.callbacks.ModelCheckpoint(
                '{}/{}_{}_loss_model.h5'.format(dir_name, model.name,
                                        'model')
                    , save_weights_only=True, save_best_only=True, mode='min'
                    , monitor='val_loss'),
    tf.keras.callbacks.EarlyStopping(patience=10, restore_best_weights=False # to stop the model overfitting
                                , mode='max', verbose=1, monitor='seq_accuracy'),
             
    # tf.keras.callbacks.ReduceLROnPlateau(monitor='val_word_accuracy', mode='max',
    #                                   factor=0.5, patience=15, min_lr=0.0001, verbose=1)
    # XTensorBoard(log_dir=str(args.model_dir), **config['tensorboard'])
    tf.keras.callbacks.TensorBoard(log_dir='logs_Multi_task', profile_batch = 0 #'10, 25'
                                    )
    # , DecayHistory()
    ]

model.compile(optimizer=optimizer,
                    loss=loss_funct, 
                    metrics=[seq_accuracy, edit_dist]
              )

## Training steps

In [ ]:
start_train = time.time() 

history = model.fit(train_loader, epochs=EPOCHS, callbacks=callbacks,
                    validation_data=val_loader
                    )

print('[TRAINING FINISHED]\n'
                'Model trained during {} \n'.format( asHHMMSS( time.time() - start_train)))

In [ ]:
model.evaluate(val_loader)

In [ ]:
model.load_weights('{}/{}_{}_best_model.h5'.format(dir_name, model.name,
                                        'model'))   # 1s 48ms/step - loss: 0.2793 - seq_accuracy: 0.7572 - edit_dist: 0.0804 # relu
model.evaluate(val_loader)                          # 1s 57ms/step - loss: 0.2883 - seq_accuracy: 0.7871 - edit_dist: 0.0753 # swish
                                                    # 1s 56ms/step - loss: 0.3118 - seq_accuracy: 0.7584 - edit_dist: 0.0794 # bad swish
                                                    # 2s 63ms/step - loss: 0.2998 - seq_accuracy: 0.7436 - edit_dist: 0.0841 # bad swish

In [ ]:
model.load_weights('{}/{}_{}_loss_model.h5'.format(dir_name, model.name,
                                        'model'))   # 1s 32ms/step - loss: 0.2756 - seq_accuracy: 0.7572 - edit_dist: 0.0756
model.evaluate(val_loader)  

In [ ]:
# Load the TensorBoard notebook extension.
%load_ext tensorboard
%tensorboard --logdir logs_CNNs/

### Visualization of the training

In [ ]:
plt.style.use('ggplot')

def plot_loss(hist):
    plt.figure(figsize=(14, 7))

    plt.plot(hist['loss'], label='Train')
    plt.plot(hist['val_loss'], label='Valid')

    plt.annotate(f"Epoch: { np.argmin(hist['val_loss']) }; \nMin valid-loss: {min(hist['val_loss']) :.4f}"
                , xy =( np.argmin(hist['val_loss'] ) , min( hist['val_loss'] ) ), 
                textcoords="offset points", # how to position the text
                 xytext=( -73 , 50), # distance from text to points (x,y)
                 ha='center', # horizontal alignment can be left, right or center
                arrowprops = dict(facecolor ='green', 
                                  shrink = 0.05), fontsize=14
            )
    plt.xlabel('Epochs', fontsize=16)
    plt.ylabel('Loss', fontsize=16)
    plt.legend(fontsize=16)

plot_loss(history.history)

In [ ]:
history.history

In [ ]:
def plot_evaluations(hist):
    f, (ax1, ax2) = plt.subplots(1, 2, figsize=(26, 7))

    ax1.plot(hist['seq_accuracy'],  '--', label='Train')
    ax1.plot(hist['val_seq_accuracy'],  '-', label='Valid')
    ax1.plot( np.argmax(hist['val_seq_accuracy']), max(hist['val_seq_accuracy']), linestyle="None", color="red", marker="X"
        , markersize=10, label='Max Seq acc')
    
    ax1.annotate(f"Epoch: {np.argmax(hist['val_seq_accuracy'])}; \nSeq acc: {max(hist['val_seq_accuracy']) :.4f}"
                , xy =(np.argmax(hist['val_seq_accuracy']) , max(hist['val_seq_accuracy'])), 
                textcoords="offset points", # how to position the text
                #  xytext=( 5 , 50), # distance from text to points (x,y)
                 xytext=( -32 , -50), # distance from text to points (x,y)
                 ha='center', # horizontal alignment can be left, right or center
                # arrowprops = dict(facecolor ='green', 
                #                   shrink = 0.05)
                # va='bottom',
                fontsize=14
             ) 
    ax1.set_title(label='Sequence accuracy', fontsize=16)
    ax1.set_xlabel('Epochs', fontsize=16)
    ax1.legend(fontsize=16)

    ax2.plot(hist['edit_dist'],  '--',  label='Train')
    ax2.plot(hist['val_edit_dist'],  '-', label='Valid')
    ax2.plot( np.argmin(hist['val_edit_dist']), min(hist['val_edit_dist']), linestyle="None", color="red", marker="X"
        , markersize=10, label=r'Min Edit dist')
    ax2.annotate(f"Epoch: {np.argmin(hist['val_edit_dist'])}; \nEdit distance: {min(hist['val_edit_dist']) :.4f}"
                , xy =(np.argmin(hist['val_edit_dist']) , min(hist['val_edit_dist'])), 
                textcoords="offset points", # how to position the text
                #  xytext=( 5 , 50), # distance from text to points (x,y)
                 xytext=( -58 , 20), # distance from text to points (x,y)
                 ha='center', # horizontal alignment can be left, right or center
                fontsize=14
             ) 

    ax2.set_title(label='Edit distance', fontsize=16)
    ax2.set_xlabel('Epochs', fontsize=16)
    ax2.legend(fontsize=16)

plot_evaluations(history.history)

## Predictions

In [ ]:
for i in range(len(inp)):
    idx = i

   
    img = tf.expand_dims(inp[idx], 0)
    y_pred = model(img)

    print(f"\nImage {i+1}:")
    y_pred = map2string( np.argmax(y_pred, axis=-1) )

    visualize(
        # image=inp[idx][..., 0]
        image= denormalize( inp[idx].numpy() )
        # , colormap='gray'
        , title=f'Multi-task prediction: {y_pred}'
    )
    print(f'Greedy pred: {y_pred}')
    print(f"Actual targ: { map2string( np.argmax(tar[i], axis=-1) ) } " )

In [ ]:
references = []
hypothesis = []
bleu = []
ctr, all_acc = 0, 0

for batch, (inp, tar) in enumerate(val_loader):
    for i in range(inp.shape[0]):
        img = tf.expand_dims(inp[i], 0)
        y_pred = model(img)
        # y_pred = decoder.decode(y_pred, method='greedy')[0]
        y_pred = map2string( np.argmax(y_pred, axis=-1) )
        y_target = map2string( np.argmax(tar[i], axis=1) )

        references.append(y_target)
        hypothesis.append(y_pred)
        # weights = ( 1./5, 1./5, 1./5, 1./5, 1./5 )
        # weights = ( 1./3, 1./3, 1./3 )        
        # weights = ( 1./2, 1./2 )        
        weights = ( 1./4, 1./4, 1./4, 1./4 )        
        bleu.append( sentence_bleu([list( y_target)], list( y_pred ),  weights=weights, smoothing_function=smoothier().method1))
        if y_target.split(".")[0] == y_pred.split('.')[0]:
            ctr+=1
        if y_target == y_pred:
            all_acc+=1

print(f"Left acc: {(ctr/len(references) * 100):.4f} \tBLEU acc: {(np.mean(bleu) * 100):.4f} \tAll acc: {(all_acc/len(references)*100):.4f}")
# Left acc: 82.9201 	BLEU acc: 86.6904 	All acc: 71.9008
# Left acc: 81.8182 	BLEU acc: 85.2246 	All acc: 66.1157
# Left acc: 86.7769 	BLEU acc: 87.6802 	All acc: 73.0028
# Left acc: 87.6033 	BLEU acc: 89.4241 	All acc: 76.0331 # using color =)
# Left acc: 88.4298 	BLEU acc: 89.8931 	All acc: 78.7879 # using swish

# HPARAMS

In [ ]:
from tensorflow.keras.layers import *
from tensorflow.keras.regularizers import L2

tf.keras.backend.clear_session()

def build_model(nb_classes, hparams=None, shape=(64, 288, 1), weight_decay=0
                , batch_momentum=0.99):
    # ACTIV= hparams[HP_ACTIVATION] #"swish"
    ACTIV= hparams[HP_ACTIVATION] if hparams else "swish"
    
    bn_axis = 3
    input_ = tf.keras.layers.Input(shape=shape)
    x = Conv2D(128, (3, 3), padding='same', use_bias=False, name='block1_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay) )(input_)
    x = BatchNormalization(axis=bn_axis, name='bn00_x1', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)

    x = Conv2D(128, (3, 3), padding='same', use_bias=False, name='block1_conv2', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn01_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block1_pool')(x)

    # # Block 2
    x = Conv2D(128, (3, 3), padding='same', use_bias=False, name='block2_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn11_x1', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)

    x = Conv2D(128, (3, 3), padding='same', use_bias=False, name='block2_conv2', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn12_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block2_pool')(x)

    # # Block 3
    x = Conv2D(256, (3, 3), padding='same', use_bias=False, name='block3_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn21_x1', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)

    x = Conv2D(256, (3, 3), padding='same', use_bias=False, name='block3_conv2', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn22_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block3_pool')(x)

    # Block 4
    x = Conv2D(512, (3, 3), padding='same', use_bias=False, name='block4_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn31_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    
    x = Conv2D(512, (3, 3), padding='same', use_bias=False, name='block4_conv2', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn32_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block4_pool')(x)
    
    # Block 5
    x = Conv2D(512, (3, 3), padding='same', use_bias=False, name='block5_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn41_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    
    x = Conv2D(1024, (3, 3), padding='same', use_bias=False, name='block5_conv3', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn43_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    x = MaxPooling2D((2, 2), strides=(2, 2), name='block5_pool')(x)
    
    # Block 6
    x = Conv2D(1024, (3, 3), padding='same', use_bias=False, name='block6_conv1', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn51_x2', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    
    x = Conv2D(1024*2, (3, 3), padding='same', use_bias=False, name='block6_conv12', kernel_initializer='he_normal', kernel_regularizer=L2(weight_decay))(x)
    x = BatchNormalization(axis=bn_axis, name='bn51_x22', momentum=batch_momentum)(x)
    x = Activation(ACTIV)(x)
    x = tf.keras.layers.Dropout(0.3)(x)


    # Max-pooling layers are worse at preserving localization --> AveragePooling fits here better
    X = AveragePooling2D((2, 2), strides=(2, 1), name='avg_pool1', padding ='same')(x)
    

    X = tf.keras.layers.Reshape((-1, 1024*2), name='Reshape')(X)
    # block7     
    X = Conv1D(512, 3, strides=1, padding='same', name = 'conv1y', activation=None,  dilation_rate=1,  use_bias=False, kernel_initializer="he_normal", kernel_regularizer=L2(0.000))(X)
    X = BatchNormalization(axis=2, name = 'bn01y')(X)
    X = Activation(ACTIV)(X)
    X = tf.keras.layers.Dropout(0.3)(X)
    X = tf.keras.layers.Conv1D(filters=nb_classes, kernel_size=1 , strides=1, padding='same', name = 'conv1x', activation=None,  dilation_rate=1,  use_bias=False, kernel_initializer="he_normal")(X)
    X = tf.keras.layers.BatchNormalization(axis=2, name = 'bnhe')(X)
    X = tf.keras.layers.Activation('softmax', name='Softmax')(X)

    model = tf.keras.Model(inputs=input_, outputs=X, name='Multi_task')
    return model

In [ ]:
from tensorboard.plugins.hparams import api as hp

# HP_BATCH_SIZE = hp.HParam('batch_size', hp.Discrete([8, 16 , 24, 32]))
HP_BATCH_SIZE = hp.HParam('batch_size', hp.Discrete([ 16 ]))

# HP_LR_RATE = hp.HParam('lr_rate', hp.Discrete([0.0005 , 0.001 ])) 
HP_LR_RATE = hp.HParam('lr_rate', hp.Discrete([0.0005 , 0.00055, 0.0006 ])) # 0.0006 is the best
# HP_LR_RATE = hp.HParam('lr_rate', hp.RealInterval(0.0005, 0.00075))
# HP_ACTIVATION = hp.HParam('activation', hp.Discrete(['relu', 'swish'])) 

# METRIC_ACCURACY = 'seq-accuracy'

HPARAM_path = 'logs/hparam_tuning'
HPARAMS = [HP_BATCH_SIZE, HP_LR_RATE,
        #    HP_ACTIVATION
           ]
METRICS = [hp.Metric("seq_accuracy", display_name='Seq Accuracy'),
        #    hp.Metric('seq_accuracy', group='train')
           ]

# with tf.summary.create_file_writer(HPARAM_path).as_default():
#   hp.hparams_config(
#     hparams=[HP_BATCH_SIZE], #, HP_DROPOUT, HP_OPTIMIZER],
#     metrics=[hp.Metric(METRIC_ACCURACY, display_name='Accuracy')],
#   )

In [ ]:
HP_LR_RATE = hp.HParam('lr_rate', hp.RealInterval(0.0005, 0.00075))

for lr_rate in tf.linspace(
      HP_LR_RATE.domain.min_value,
      HP_LR_RATE.domain.max_value,
      4,): 
      print(lr_rate.numpy())

In [ ]:
print('Path:', HPARAM_path)

def run(base_logdir, session_id, hparams):
    """Run a training/validation session.
    Flags must have been parsed for this function to behave.
    Args:
      base_logdir: The top-level logdir to which to write summary data.
      session_id: A unique string ID for this session.
      hparams: A dict mapping hyperparameters in `HPARAMS` to values.
    """
    model = build_model(nb_classes=data_builder.num_classes.numpy(), shape=(IMG_HEIGHT, IMG_WIDTH, IMG_CHANNELS)
                        # , hparams=hparams
                        , hparams=None
                        ) 
    logdir = os.path.join(base_logdir, session_id)
    # dataloaders
    train_loader = data_builder.build(X_cut_train, y_train, hparams[HP_BATCH_SIZE], is_training=True)
    val_loader = data_builder.build(X_cut_valid, y_valid, hparams[HP_BATCH_SIZE], False)

    callback = tf.keras.callbacks.TensorBoard(
        logdir,
        profile_batch=0,  # workaround for issue #2084
    )
    hparams_callback = hp.KerasCallback(logdir, hparams) 
    callbacks = [callback, hparams_callback,
                 tf.keras.callbacks.ModelCheckpoint(
                '{}/{}_{}_best_model.h5'.format(logdir, model.name,
                                        'model')
                    , save_weights_only=True, save_best_only=True, mode='max'
                    , monitor='val_seq_accuracy')
                 , tf.keras.callbacks.EarlyStopping(
                        monitor='val_seq_accuracy', patience=12,
                        mode='max', restore_best_weights=True
                        )
                 
                ]
    # optimizer = tf.keras.optimizers.Adam(0.0005, clipnorm=CLIP_NORM)
    optimizer = tf.keras.optimizers.Adam(hparams[HP_LR_RATE], clipnorm=CLIP_NORM)
    model.compile(optimizer=optimizer,
                    loss=loss_funct, 
                    metrics=[seq_accuracy, edit_dist]
              )
    # training
    model.fit(train_loader, verbose=0, epochs=100, callbacks=callbacks,
                    validation_data=val_loader
                    )
    # loading best weights
    # model.load_weights('{}/{}_{}_best_model.h5'.format(logdir, model.name,
    #                                     'model'))
    # pass over the valid set - plot those values
    loss, seq_acc, edit_dt = model.evaluate(val_loader)
    with tf.summary.create_file_writer(logdir).as_default():
        hp.hparams(hparams)  # record the values used in this trial
        # grab the last value of the pass
        # seq_acc = result.history['val_seq_accuracy'][-1]
        tf.summary.scalar('seq_accuracy', seq_acc, step=1)

In [ ]:
tf.keras.backend.clear_session()

def run_all(logdir, verbose=False):
    """Perform random search over the hyperparameter space.
    Arguments:
      logdir: The top-level directory into which to write data. This
        directory should be empty or nonexistent.
      verbose: If true, print out each run's name as it begins.
    """

    with tf.summary.create_file_writer(logdir).as_default():
        hp.hparams_config(hparams=HPARAMS, metrics=METRICS)
    
    sessions_per_group = 3
    num_session_groups = 1
    num_sessions = num_session_groups * sessions_per_group
    session_index = 0  # across all session groups
    # for group_index in range(num_session_groups):   # to have diff groups     
    for batch_sz in HP_BATCH_SIZE.domain.values:
        for lr_rate in HP_LR_RATE.domain.values:
        # for lr_rate in tf.linspace(HP_LR_RATE.domain.min_value,
        #                                 HP_LR_RATE.domain.max_value, 4):
                # lr_rate = lr_rate.numpy()
            # for act in HP_ACTIVATION.domain.values:
                hparams = {HP_BATCH_SIZE: batch_sz, HP_LR_RATE: lr_rate
                        #    , HP_ACTIVATION: act
                           }
                for repeat_index in range(sessions_per_group): # to run several times same feature
                    # session_id = f'run_{session_index}--{batch_sz}-{lr_rate}'
                    session_id = f'run_{session_index}-{repeat_index}--{batch_sz}-{lr_rate}'
                    if verbose:
                        # print(f'---Running training session {session_index}/{num_sessions}')
                        # print(f'---Running training: run-{session_index}')
                        print(f'---Running training: run-{session_index}--{repeat_index + 1}')
                        print({h.name: hparams[h] for h in hparams})
                        # print(f'--- repeat # {repeat_index + 1}')
                    run(
                        base_logdir=logdir,
                        session_id=session_id,
                        hparams=hparams,
                    )
                session_index += 1

tf.random.set_seed( SEED )               
run_all(HPARAM_path, verbose=True)


In [ ]:
# past run

In [ ]:
%reload_ext  tensorboard
%tensorboard --logdir logs/hparam_tuning

In [ ]:
%reload_ext  tensorboard
%tensorboard --logdir logs/hparam_tuning

In [ ]:
# 1st pass checking last dropOUt layer

In [ ]:
# Load the TensorBoard notebook extension.
%reload_ext  tensorboard
%tensorboard --logdir logs/hparam_tuning

In [ ]:
# !rm -r logs/

### so far the best is 8 and 16 with 0.0005 and swish

In [ ]:
# Load the TensorBoard notebook extension.
%reload_ext  tensorboard
%tensorboard --logdir logs/hparam_tuning

In [ ]:
# !ps ax | grep tensorboard # tensorboard --logdir logs/hparam_tuning
# !kill 4405        

# WEIGHTS

In [ ]:
!ls "{weights_dir}"
!ls "{loading_dir}"

## Saving weights

- Copying files from Colab to DRIVE

In [ ]:
!cp -vr Multi_task_model "{weights_dir}"
!cp -vr logs_Multi_task "{weights_dir}"

## Loading weights

- Copying files from Drive to COLAB

In [ ]:
!cp -vr "{weights_dir}/." ./
!ls

In [ ]:
model.built = True
model.load_weights('{}/{}_{}_best_model.h5'.format(loading_dir, model.name,
                                        'model'))

In [ ]:
model.evaluate(val_loader)

In [ ]:
model.evaluate(val_loader)

# Port to Tensorflow Lite

## Buggy TFLITE

In [ ]:
tf.saved_model.save(model, checks_dir)

!ls "{checks_dir}"

In [ ]:
converter = tf.lite.TFLiteConverter.from_saved_model(checks_dir)
# converter = tf.lite.TFLiteConverter.from_keras_model(model)

In [ ]:
converter.optimizations = [tf.lite.Optimize.DEFAULT]
# converter.experimental_new_converter = True
converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS, # enable TensorFlow Lite ops.
        tf.lite.OpsSet.SELECT_TF_OPS # enable TensorFlow ops.
    ]

In [ ]:
converter.target_spec.supported_types = [tf.float16]
tf_lite_model = converter.convert()
# saving the model
open(f'multi_detector_tf_float16.tflite', 'wb').write(tf_lite_model)
# ------------------------------------
# converter.target_spec.supported_types = [tf.float32]
# tf_lite_model = converter.convert()
# # saving the model
# open(f'multi_detector_tf_float32.tflite', 'wb').write(tf_lite_model)

In [ ]:
!du -sh multi_detector_tf_float16.tflite

## Another transformation approach

Useful links:
- https://www.javaer101.com/es/article/54369523.html
- http://5.9.10.113/64850356/error-with-strided-slice-in-tensorflow-lite

In [ ]:
input_shape = model.inputs[0].shape.as_list()
input_shape[0] = 1 # batch_size: the model will predict image-per-image

func = tf.function(model).get_concrete_function(
    tf.TensorSpec(input_shape, model.inputs[0].dtype))

converter = tf.lite.TFLiteConverter.from_concrete_functions([func])
converter.optimizations = [tf.lite.Optimize.DEFAULT]
# converter.experimental_new_converter = True

converter.target_spec.supported_ops = [
        tf.lite.OpsSet.TFLITE_BUILTINS, # enable TensorFlow Lite ops.
        tf.lite.OpsSet.SELECT_TF_OPS # enable TensorFlow ops.
    ]

converter.target_spec.supported_types = [tf.float16]
tflite_model = converter.convert()

open(f'multi_detector_tf_float16.tflite', 'wb').write(tflite_model)
!du -sh multi_detector_tf_float16.tflite

In [ ]:
converter.target_spec.supported_types = [tf.float32]
tflite_model = converter.convert()

open(f'multi_detector_tf_float32.tflite', 'wb').write(tflite_model)
!du -sh multi_detector_tf_float32.tflite

In [ ]:
def infere_tf_lite_model(path, image):
    input_data = image

    interpreter = tf.lite.Interpreter(model_path=path)
    interpreter.allocate_tensors()

    # Get input and output tensors.
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_shape = input_details[0]['shape']
    interpreter.set_tensor(input_details[0]['index'], input_data)
    interpreter.invoke()

    output = interpreter.get_tensor(output_details[0]['index'])
    return output


#  grab image from valid dataset
print(f"Batch has {inp.shape[0]} images.")
idx = 3
image = np.expand_dims(inp[idx], axis=0)

#  predict sequence-of-numbers
output = infere_tf_lite_model("multi_detector_tf_float16.tflite", image=image)

y_pred = map2string( np.argmax(output, axis=-1) )
# plots
visualize(
    image= denormalize(inp[idx].numpy())
    , title=f'Multi-task prediction: {y_pred}'
)

# print(f"Image: {idx}; predicted sequence: {y_pred}")


In [ ]:
# copy LITE model to DRIVE
!cp -vr multi_detector_tf_float16.tflite "{weights_dir}"
!cp -vr multi_detector_tf_float32.tflite "{weights_dir}"

In [ ]:

tests = ["00883201372079_0.jpg", "02800000312555_0.jpg", "00882401101858_0.jpg", "00882401146713_0.jpg", "00883201360504_0.jpg" ]

test_set = []
for t in tests: # use funct inside data builder
    test, _ = data_builder._decode_img(t, label=None)
    test_set.append(test)
# predicting the images    
for test in test_set:
    image = np.expand_dims(test.numpy(), axis=0)
    #  predict sequence-of-numbers
    output = infere_tf_lite_model("multi_detector_tf_float16.tflite", image=image)

    y_pred = map2string( np.argmax(output, axis=-1) )
    # plots
    visualize(
        image= denormalize(image[0])
        , title=f'Multi-task prediction: {y_pred}'
    )
    print(f"Predicted sequence: {y_pred}")

# Model summary

In [ ]:
model.summary()

# others

In [ ]:
# inp = tf.random.uniform(shape=(1, 40,160,3))
inp = tf.random.uniform(shape=(1, 64, 288, 1))


In [ ]:
pred = model.predict(inp)
print(pred.shape)
pred = np.argmax(pred, axis=-1)

pred

In [ ]:
letters = " ABCDEFGHIJKLMNPQRSTUVWXYZ0123456789"
dic = {}
for i in range(len(letters)):
    dic[i] = letters[i]
invert_dic = {}
for i in range(len(letters)):
    invert_dic[letters[i]] = i

In [ ]:
len(letters)

In [ ]:
arr = []
for i in range(9):
   arr.append(dic[pred.squeeze()[i]])
print(arr)   

In [ ]:

# tf.keras.backend.clear_session()
# def feature_extractor(input_tensor):
#     act = 'relu'
#     x = tf.keras.layers.Conv2D(
#         128,  (5, 5), padding="same", activation=act, name='conv1')(input_tensor)
#     x = tf.keras.layers.MaxPool2D(pool_size=2, strides=2,  padding='valid', name='pool1')(x)
#     x = tf.keras.layers.Conv2D(
#         128,  (3, 3), padding="same", activation=act, name='conv2')(x)
#     x = tf.keras.layers.Conv2D(
#         192,  (3, 3), padding="same", activation=act, name='conv3')(x)
#     x = tf.keras.layers.MaxPool2D(pool_size=2, strides=2,  padding='valid', name='pool2')(x)
#     x = tf.keras.layers.Conv2D(
#         256,  (3, 3), padding="same", activation=act, name='conv4')(x)
#     x = tf.keras.layers.MaxPool2D(pool_size=2, strides=2,  padding='valid', name='pool3')(x)
#     x = tf.keras.layers.Flatten()(x)
#     return x

# def char_recogn(input, n_char, num_classes=11):
#     x = tf.keras.layers.Dense(units=512, name=f'FcChar{n_char}-1')(input)
#     x = tf.keras.layers.Dense(units=num_classes, name=f'FcChar{n_char}-2')(x)
#     x = tf.nn.softmax(x)
#     return x

# def build_model(num_classes, img_width=220, img_channels=1, img_height=60):
#     """build Multi-task model"""

#     # img_input = tf.keras.Input(shape=(img_height, img_width, img_channels))
#     img_input = tf.keras.Input(shape=(img_width, img_height, img_channels))
#     x = feature_extractor(img_input)

#     x = tf.keras.layers.Dense(units=4096, name='Fc1')(x)

#     x1 = char_recogn(x, 1)
#     x2 = char_recogn(x, 2)
#     x3 = char_recogn(x, 3)
#     x4 = char_recogn(x, 4)
#     x5 = char_recogn(x, 5)
#     x6 = char_recogn(x, 6)
#     x7 = char_recogn(x, 7)
#     x8 = char_recogn(x, 8)


#     print(x1.shape, tf.concat([x1,x2], axis=0))
#     # x = tf.concat([x1, x2, x3])
#     # x = tf.keras.layers.Dense(units=512, name='Fc2')(x)
#     # x = tf.keras.layers.Dense(units=num_classes, name='Fc3')(x)
#     return tf.keras.Model(inputs=img_input, outputs=[x1, x2, x3], name='Multi-task')

# model = build_model(num_classes=10) 
# model.summary()